# MODEL 2: GRADIENT BOOSTING
# Bank Marketing Dataset

In [1]:
# import required Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
# 1. LOAD DATASET
df = pd.read_csv("C:/Users/ASUS/Documents/bank-full.csv", sep=";")
print("Shape:", df.shape)

Shape: (45211, 17)


In [3]:
# 2. BASIC DATA CHECKS

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nTarget distribution:")
print(df["y"].value_counts())

print("\nTarget distribution (%):")
print(df["y"].value_counts(normalize=True) * 100)



Missing values:
0

Duplicate rows:
0

Target distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target distribution (%):
y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64


In [4]:
# 3. ENCODE TARGET VARIABLE


df["y"] = df["y"].map({
    "no": 0,
    "yes": 1
})

print("\nEncoded target distribution:")
print(df["y"].value_counts())



Encoded target distribution:
y
0    39922
1     5289
Name: count, dtype: int64


In [5]:
# 4. REMOVE TARGET AND POTENTIAL DATA LEAKAGE

X = df.drop(columns=["y", "duration"])

y = df["y"]

print("\nFeatures used:")
print(X.columns.tolist())


Features used:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


In [6]:
# 5. DEFINE NUMERICAL FEATURES

numeric_features = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous"
]


In [7]:
# 6. DEFINE CATEGORICAL FEATURES

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome"
]


In [8]:
# 7. PREPROCESSING PIPELINE

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    sparse_threshold=0
)


In [9]:
# 8. GRADIENT BOOSTING MODEL

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=5,
    subsample=0.8,
    random_state=42
)


In [10]:
# 9. COMPLETE MODEL PIPELINE

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", gb_model)
    ]
)

In [11]:
# 10. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)


Training set: (31647, 15)
Testing set: (13564, 15)


In [12]:
# 11. CALCULATE BALANCED SAMPLE WEIGHTS


sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print("\nBalanced sample weights calculated.")


Balanced sample weights calculated.


In [13]:
# 12. TRAIN GRADIENT BOOSTING MODEL

print("\nTraining Gradient Boosting model...")

model.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weights
)

print("Gradient Boosting training completed.")


Training Gradient Boosting model...
Gradient Boosting training completed.


In [14]:
# 13. GENERATE PREDICTIONS

y_pred_gb = model.predict(X_test)

y_prob_gb = model.predict_proba(X_test)[:, 1]

print("\nPredictions generated successfully.")

print("Unique values in y_test:", np.unique(y_test))
print("Unique values in y_pred:", np.unique(y_pred_gb))


Predictions generated successfully.
Unique values in y_test: [0 1]
Unique values in y_pred: [0 1]


In [15]:
# 14. CALCULATE PERFORMANCE METRICS


accuracy_gb = accuracy_score(y_test, y_pred_gb)

precision_gb = precision_score( y_test, y_pred_gb,pos_label=1,zero_division=0)

recall_gb = recall_score( y_test,y_pred_gb,pos_label=1,zero_division=0)

f1_gb = f1_score( y_test,y_pred_gb,pos_label=1,zero_division=0)

roc_auc_gb = roc_auc_score(y_test,y_prob_gb)

cm_gb = confusion_matrix( y_test,y_pred_gb)


In [16]:
# 15. DISPLAY RESULTS

print("\n" + "=" * 60)
print("GRADIENT BOOSTING RESULTS")
print("=" * 60)

print(f"Accuracy:  {accuracy_gb:.4f}")
print(f"Precision: {precision_gb:.4f}")
print(f"Recall:    {recall_gb:.4f}")
print(f"F1 Score:  {f1_gb:.4f}")
print(f"ROC-AUC:   {roc_auc_gb:.4f}")

print("\nConfusion Matrix:")
print(cm_gb)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred_gb,
        target_names=[
            "No Subscription",
            "Subscription"
        ],
        zero_division=0
    )
)


GRADIENT BOOSTING RESULTS
Accuracy:  0.7965
Precision: 0.3189
Recall:    0.6509
F1 Score:  0.4281
ROC-AUC:   0.7960

Confusion Matrix:
[[9771 2206]
 [ 554 1033]]

Classification Report:
                 precision    recall  f1-score   support

No Subscription       0.95      0.82      0.88     11977
   Subscription       0.32      0.65      0.43      1587

       accuracy                           0.80     13564
      macro avg       0.63      0.73      0.65     13564
   weighted avg       0.87      0.80      0.82     13564

